In [3]:
import os
print(os.getcwd())

C:\Users\leopa\AppData\Local\Programs\Antigravity


# Step 1: Data Preparation

In this notebook, we:
1. Read the `reviews83325.csv` and `Tripadvisor.csv` databases.
2. Filter reviews to keep only English reviews to reduce vocabulary entropy.
3. Group variable number of reviews per place.
4. Use Term Frequency-Inverse Document Frequency (TF-IDF) to extract the best 100 words summarizing a place, acting as our normalization mechanism.

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
import warnings
warnings.filterwarnings('ignore')

### 1. Load Datasets & EDA

In [2]:
df_reviews = pd.read_csv('../data/reviews83325.csv')
df_places = pd.read_csv('../data/Tripadvisor.csv')

print(f"Total reviews: {len(df_reviews)}")
print(f"Total places: {len(df_places)}")

FileNotFoundError: [Errno 2] No such file or directory: '../data/reviews83325.csv'

### 2. Filter English Reviews

In [12]:
df_reviews_en = df_reviews[df_reviews['langue'] == 'en'].copy()
print(f"English reviews: {len(df_reviews_en)}")

English reviews: 153071


### 3. Group Reviews by Place

In [13]:
df_grouped = df_reviews_en.groupby('idplace')['review'].apply(lambda x: ' '.join(x.dropna().astype(str))).reset_index()
print(f"Places with English reviews: {len(df_grouped)}")

Places with English reviews: 1835


### 4. TF-IDF Normalization (Top 100 Words)

### 4. TF-IDF Normalization (Top 100 Words) — Corpus-Aware

In [14]:
# FIX: The vectorizer MUST be fit on the ENTIRE corpus first so that the
# Inverse Document Frequency (IDF) is computed across all 1835 places.
# Previously the vectorizer was fit on each single document, making IDF=1
# for every term (i.e. pure term frequency, not TF-IDF).
#
# Correct approach:
#   1. Fit TfidfVectorizer on ALL concatenated review strings at once.
#   2. For each individual place, transform its text and pick the top-100
#      terms by TF-IDF score — words frequent in *this* place but rare
#      across the corpus (true discriminative keywords).

corpus_vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
all_texts = df_grouped['review'].fillna('').tolist()
corpus_vectorizer.fit(all_texts)
feature_names = corpus_vectorizer.get_feature_names_out()

def extract_top_100_tfidf(text):
    if not isinstance(text, str) or len(text.strip()) == 0:
        return ""
    try:
        X = corpus_vectorizer.transform([text])
        scores = list(zip(feature_names, X.toarray().ravel()))
        sorted_scores = sorted(scores, key=lambda x: x[1], reverse=True)
        top_words = [w for w, s in sorted_scores if s > 0][:100]
        return " ".join(top_words) if top_words else text
    except ValueError:
        return text

df_grouped['top_100_words'] = df_grouped['review'].apply(extract_top_100_tfidf)
print(f"Sample keywords for place 0: {df_grouped['top_100_words'].iloc[0][:80]}...")
df_grouped.head()


Sample keywords for place 0: square park place hugo vosges victor des galleries beautiful paris marais buildi...


,idplace,review,top_100_words
0,188467,Personally I think it is the most beautiful sq...,square park place hugo vosges victor des galle...
1,188468,My old college friend and I booked this beauti...,shopping street apartment marais earrings des ...
2,188470,"Being winter and all, not a lot going on, howe...",village shops paul st antiques antique courtya...
3,188471,To call Au Passe Partout a shop is a serious u...,history owner unique keys marked specialty ext...
4,188472,Very old historical place. I attended to exper...,church des archives arches exhibitions art pla...


### 5. Format and Export the Prepared Dataset

In [15]:
# Save the prepared data to a new CSV file to inject directly into the ML models
df_final = df_grouped[['idplace', 'top_100_words', 'review']]
df_final.to_csv('prepared_reviews.csv', index=False)
print("Cleaned reviews dataset saved as 'prepared_reviews.csv'")

Cleaned reviews dataset saved as 'prepared_reviews.csv'
